### Intro
- Implementation of RNN, GRU and LSTM from scratch, using pytorch
- PyTorch (pre-0.4), to track gradients, you had to wrap tensors in Variable. fastai=0.7 wraps pytorch Variable with a helper to V.
- pytorch(0.4) deprecated Variable and merged with tensor

In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

from fastai.io import*
from fastai.conv_learner import*

from fastai.column_data import*

### Set up

##### Cuda device setup

In [2]:
import torch, torchvision
print("torch:", torch.__version__)
print("CUDA toolkit:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
print("torchvision:", torchvision.__version__)
print(torch.backends.cudnn.version())

torch.cuda.set_device(0)
torch.backends.cudnn.enabled = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current CUDA device:", torch.cuda.current_device())
    print("CUDA device name:", torch.cuda.get_device_name(0))

torch: 0.3.1
CUDA toolkit: 9.0.176
GPU available: True
torchvision: 0.2.1
7005
CUDA available: True
CUDA device count: 1
Current CUDA device: 0
CUDA device name: Tesla T4


In [3]:
PATH = 'data/nietzsche/'

In [4]:
get_data("https://s3.amazonaws.com/text-datasets/nietzsche.txt", f'{PATH}nietzsche.txt')
text = open(f'{PATH}nietzsche.txt').read()
print('corpus length:', len(text))

corpus length: 600893


In [5]:
text[:400]

'PREFACE\n\n\nSUPPOSING that Truth is a woman--what then? Is there not ground\nfor suspecting that all philosophers, in so far as they have been\ndogmatists, have failed to understand women--that the terrible\nseriousness and clumsy importunity with which they have usually paid\ntheir addresses to Truth, have been unskilled and unseemly methods for\nwinning a woman? Certainly she has never allowed herself '

In [6]:
chars = sorted(list(set(text)))
vocab_size = len(chars)+1
print('total chars: ', vocab_size)

total chars:  85


- add "\0" as a padding char at the start

In [7]:
chars.insert(0, "\0")

In [8]:
print(chars)

['\x00', '\n', ' ', '!', '"', "'", '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'Æ', 'ä', 'æ', 'é', 'ë']


In [9]:
''.join(chars[1:-6])

'\n !"\'(),-.0123456789:;=?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxy'

Map from chars to indices and back indices to chars

In [10]:
char_indices = dict((c, i) for i, c in enumerate(chars))
indices_char = dict((i, c) for i, c in enumerate(chars))

In [11]:
idx = [char_indices[c] for c in text]

In [12]:
idx[:10]

[40, 42, 29, 30, 25, 27, 29, 1, 1, 1]

In [13]:
#reverse checking those index indices gives the characters
''.join(indices_char[i] for i in idx[:70])

'PREFACE\n\n\nSUPPOSING that Truth is a woman--what then? Is there not gro'

##### 2 Three Character Model
##### 2.1 Create inputs

Create a list of every 4th charc, starting at 0th,1st,2nd,then 3rd charct

In [14]:
cs=3
c1_dat = [idx[i] for i in range(0, len(idx)-1-cs, cs)]
c2_dat = [idx[i+1] for i in range(0, len(idx)-1-cs, cs)]
c3_dat = [idx[i+2] for i in range(0, len(idx)-1-cs, cs)]
c4_dat = [idx[i+3] for i in range(0, len(idx)-1-cs, cs)]

In [15]:
# print(''.join(indices_char[i] for i in c1_dat[:10]))
# print(''.join(indices_char[i] for i in c2_dat[:10]))
# print(''.join(indices_char[i] for i in c3_dat[:10]))
# print(''.join(indices_char[i] for i in c4_dat[:10]))

In [16]:
x1 = np.stack(c1_dat[:-2])
x2 = np.stack(c2_dat[:-2])
x3 = np.stack(c3_dat[:-2])

In [17]:
y = np.stack(c4_dat[:-2])

In [18]:
x1[:4],x2[:4],x3[:4]

(array([40, 30, 29,  1]), array([42, 25,  1, 43]), array([29, 27,  1, 45]))

In [19]:
y[:4]

array([30, 29,  1, 40])

In [20]:
x1.shape, y.shape

((200295,), (200295,))

##### Create and train model

In [21]:
n_hidden = 256

In [22]:
n_fac = 42

In [23]:
class Char3Model(nn.Module):
    def __init__(self, vocab_size, n_fac):
        super().__init__()
        self.e = nn.Embedding(vocab_size, n_fac)
        self.l_in = nn.Linear(n_fac, n_hidden)
        self.l_hidden = nn.Linear(n_hidden, n_hidden)
        self.l_out = nn.Linear(n_hidden, vocab_size)

    def forward(self, c1, c2, c3):
        in1 = F.relu(self.l_in(self.e(c1)))
        in2 = F.relu(self.l_in(self.e(c2)))
        in3 = F.relu(self.l_in(self.e(c3)))

        #make h matrix int zero for refactoring(make them identical) the next lines into a loop
        h = V(torch.zeros(in1.size()))#.cuda()) 
        h = F.tanh(self.l_hidden(h+in1))
        h = F.tanh(self.l_hidden(h+in2))
        h = F.tanh(self.l_hidden(h+in3))

        return F.log_softmax(self.l_out(h))

In [24]:
md = ColumnarModelData.from_arrays('.', [-1], np.stack([x1,x2,x3], axis=1), y, bs=512)

In [25]:
m = Char3Model(vocab_size, n_fac).cuda()

In [26]:
it = iter(md.trn_dl)
*xs, yt = next(it)
t = m(*V(xs))

In [27]:
t

Variable containing:
-4.3541 -4.0493 -4.8509  ...  -4.4532 -4.3823 -4.1355
-4.2555 -4.3483 -4.8486  ...  -4.2849 -4.2856 -4.4461
-4.4196 -4.3824 -4.5785  ...  -4.4041 -4.3827 -4.7178
          ...             ⋱             ...          
-4.2748 -4.3370 -4.9418  ...  -4.3170 -4.4235 -4.4142
-4.3977 -4.6218 -4.6874  ...  -4.2687 -4.4555 -4.6490
-4.5094 -4.5160 -4.7952  ...  -4.1164 -4.3415 -4.6100
[torch.cuda.FloatTensor of size 512x85 (GPU 0)]

In [28]:
opt = optim.Adam(m.parameters(), 1e-2)

In [32]:
fit(m, md, 1, opt, F.nll_loss)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                               
    0      2.141931   4.871233  


[array([4.87123])]

setting learning rate manually as fastai learner, lr finder is not used 

In [33]:
set_lrs(opt, 0.001)

In [34]:
fit(m, md, 1, opt, F.nll_loss)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                               
    0      1.885174   5.689183  


[array([5.68918])]

##### Test Model

In [35]:
def get_next(inp):
    idxs = T(np.array([char_indices[c] for c in inp]))
    p = m(*VV(idxs))
    i = np.argmax(to_np(p))
    return chars[i]

In [36]:
get_next('y. ')

'T'

In [37]:
get_next('ppl')

'e'

In [38]:
get_next(' th')

'e'

##### 3 First RNN
###### 3.1 Create inputs

In [29]:
cs = 8

In [30]:
c_in_dat = [[idx[i+j] for i in range(cs)] for j in range(len(idx)-cs-1)]

In [31]:
c_out_dat = [idx[j+cs] for j in range(len(idx)-cs-1)]

In [32]:
xs = np.stack(c_in_dat, axis=0)

In [33]:
xs.shape

(600884, 8)

In [34]:
y = np.stack(c_out_dat)

In [35]:
#overlapping set of 8 characters, the 1st 8th, 2nd 8th char.....
xs[:cs, :cs]

array([[40, 42, 29, 30, 25, 27, 29,  1],
       [42, 29, 30, 25, 27, 29,  1,  1],
       [29, 30, 25, 27, 29,  1,  1,  1],
       [30, 25, 27, 29,  1,  1,  1, 43],
       [25, 27, 29,  1,  1,  1, 43, 45],
       [27, 29,  1,  1,  1, 43, 45, 40],
       [29,  1,  1,  1, 43, 45, 40, 40],
       [ 1,  1,  1, 43, 45, 40, 40, 39]])

In [36]:
y[:cs]

array([ 1,  1, 43, 45, 40, 40, 39, 43])

#### Create and train model

In [37]:
val_idx = get_cv_idxs(len(idx)-cs-1)

In [38]:
md = ColumnarModelData.from_arrays('.', val_idx, xs, y, bs=512)

In [39]:
class CharLoopModel(nn.Module):
    def __init__(self, vocab_size, n_fac):
        super().__init__()
        self.e = nn.Embedding(vocab_size, n_fac)
        self.l_in = nn.Linear(n_fac, n_hidden)
        self.l_hidden = nn.Linear(n_hidden, n_hidden)
        self.l_out = nn.Linear(n_hidden, vocab_size)

    def forward(self, *cs):
        bs = cs[0].size(0)
        #print("bs size is:", bs)
        h = V(torch.zeros(bs, n_hidden))#.cuda())
        for c in cs:
            inp = F.relu(self.l_in(self.e(c)))
            h = F.tanh(self.l_hidden(h+inp))

        return F.log_softmax(self.l_out(h))

In [40]:
m = CharLoopModel(vocab_size, n_fac).cuda()
opt = optim.Adam(m.parameters(), 1e-2)

In [52]:
fit(m, md, 1, opt, F.nll_loss)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.998326   1.981383  


[array([1.98138])]

In [53]:
set_lrs(opt, 0.001)

In [54]:
fit(m, md, 1, opt, F.nll_loss)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.705428   1.712198  


[array([1.7122])]

In [41]:
class CharLoopConcatModel(nn.Module):
    def __init__(self, vocab_size, n_fac):
        super().__init__()
        self.e = nn.Embedding(vocab_size, n_fac)
        self.l_in = nn.Linear(n_fac+n_hidden, n_hidden)
        self.l_hidden = nn.Linear(n_hidden, n_hidden)
        self.l_out = nn.Linear(n_hidden, vocab_size)

    def forward(self, *cs):
        bs = cs[0].size(0)
        h = V(torch.zeros(bs, n_hidden))#.cuda()
        for c in cs:
            inp = torch.cat((h, self.e(c)), 1)
            inp = F.relu(self.l_in(inp))
            h = F.tanh(self.l_hidden(inp))

        return F.log_softmax(self.l_out(h))

In [42]:
m = CharLoopConcatModel(vocab_size, n_fac).cuda()
opt = optim.Adam(m.parameters(), 1e-3)

In [43]:
it = iter(md.trn_dl)
*xs, yt = next(it)
t = m(*V(xs))

In [58]:
fit(m, md, 1, opt, F.nll_loss)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.797361   1.777211  


[array([1.77721])]

In [59]:
set_lrs(opt,1e-4)

In [60]:
fit(m, md, 1, opt, F.nll_loss)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.681557   1.698224  


[array([1.69822])]

##### Test Model

In [61]:
def get_next(inp):
    idxs = T(np.array([char_indices[c] for c in inp]))
    p = m(*VV(idxs))
    i = np.argmax(to_np(p))
    return chars[i]

In [62]:
get_next('for thos')

'e'

In [63]:
get_next('my birth')

'e'

In [64]:
get_next('translat')

'i'

#### RNN with pytorch

In [44]:
class CharRnn(nn.Module):
    def __init__(self, vocab_size, n_fac):
        super().__init__()
        self.e = nn.Embedding(vocab_size, n_fac)
        self.rnn = nn.RNN(n_fac, n_hidden)
        self.l_out = nn.Linear(n_hidden, vocab_size)
    def forward(self, *cs):
        bs = cs[0].size(0)
        h = V(torch.zeros(1, bs, n_hidden))
        inp = self.e(torch.stack(cs))
        outp, h = self.rnn(inp, h)

        #pytorch rnn gives all the hidden state h stacked, get the last one for softmax
        return F.log_softmax(self.l_out(outp[-1]))

In [45]:
m = CharRnn(vocab_size, n_fac).cuda()
opt = optim.Adam(m.parameters(), 1e-3)

In [46]:
it = iter(md.trn_dl)
*xs, yt = next(it)

In [47]:
t = m.e(V(torch.stack(xs)))
t.size()

torch.Size([8, 512, 42])

In [48]:
ht = V(torch.zeros(1, 512, n_hidden))
outp, hn = m.rnn(t, ht)
outp.size(), hn.size()

(torch.Size([8, 512, 256]), torch.Size([1, 512, 256]))

In [49]:
t = m(*V(xs)); t.size()

torch.Size([512, 85])

In [71]:
fit(m, md, 4, opt, F.nll_loss)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.879938   1.847751  
    1      1.678648   1.67836                               
    2      1.58962    1.598258                              
    3      1.538718   1.553092                              


[array([1.55309])]

In [72]:
set_lrs(opt, 1e-4)

In [73]:
fit(m, md, 2, opt, F.nll_loss)

Epoch:   0%|          | 0/2 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.470176   1.510365  
    1      1.457546   1.505503                              


[array([1.5055])]

##### Test Model

In [74]:
def get_next(inp):
    idxs = T(np.array([char_indices[c] for c in inp]))
    p = m(*VV(idxs))
    i = np.argmax(to_np(p))
    return chars[i]

In [75]:
get_next('for thos')

'e'

In [76]:
def get_next_n(inp, n):
    res = inp
    for i in range(n):
        c = get_next(inp)
        res += c
        inp = inp[1:]+c
    return res

In [77]:
get_next_n('for thos', 45)

'for those who have the same the same the same the sam'

### Multi-output model
###### set up
lets take non-overlapping sets of characters

In [50]:
c_in_dat = [[idx[i+j] for i in range(cs)] for j in range(0, len(idx)-cs-1, cs)]

Then creat the exact same thing, offset by 1, as our labels

In [51]:
c_out_dat = [[idx[i+j] for i in range(cs)] for j in range(1, len(idx)-cs, cs)]

In [52]:
xs = np.stack(c_in_dat)
xs.shape

(75111, 8)

In [53]:
ys = np.stack(c_out_dat)
ys.shape

(75111, 8)

In [54]:
xs[:cs, :cs]

array([[40, 42, 29, 30, 25, 27, 29,  1],
       [ 1,  1, 43, 45, 40, 40, 39, 43],
       [33, 38, 31,  2, 73, 61, 54, 73],
       [ 2, 44, 71, 74, 73, 61,  2, 62],
       [72,  2, 54,  2, 76, 68, 66, 54],
       [67,  9,  9, 76, 61, 54, 73,  2],
       [73, 61, 58, 67, 24,  2, 33, 72],
       [ 2, 73, 61, 58, 71, 58,  2, 67]])

In [55]:
ys[:cs, :cs]

array([[42, 29, 30, 25, 27, 29,  1,  1],
       [ 1, 43, 45, 40, 40, 39, 43, 33],
       [38, 31,  2, 73, 61, 54, 73,  2],
       [44, 71, 74, 73, 61,  2, 62, 72],
       [ 2, 54,  2, 76, 68, 66, 54, 67],
       [ 9,  9, 76, 61, 54, 73,  2, 73],
       [61, 58, 67, 24,  2, 33, 72,  2],
       [73, 61, 58, 71, 58,  2, 67, 68]])

#### Create and train model

In [56]:
val_idx = get_cv_idxs(len(xs)-cs-1)

In [57]:
md = ColumnarModelData.from_arrays('.', val_idx, xs, ys, bs=512)

In [58]:
class CharSeqRnn(nn.Module):
    def __init__(self, vocab_size, n_fac):
        super().__init__()
        self.e = nn.Embedding(vocab_size, n_fac)
        self.rnn = nn.RNN(n_fac, n_hidden)
        self.l_out = nn.Linear(n_hidden, vocab_size)

    def forward(self, *cs):
        bs = cs[0].size(0)
        h = V(torch.zeros(1, bs, n_hidden))
        inp = self.e(torch.stack(cs))
        outp, h = self.rnn(inp, h)
        
        return F.log_softmax(self.l_out(outp), dim=-1)

In [59]:
m = CharSeqRnn(vocab_size, n_fac).cuda()
opt = optim.Adam(m.parameters(), 1e-3)

In [60]:
it = iter(md.trn_dl)
*xst, yt = next(it)

In [61]:
def nll_loss_seq(inp, targ):
    sl, bs, nh = inp.size()
    targ = targ.transpose(0,1).contiguous().view(-1)
    return F.nll_loss(inp.view(-1, nh), targ)


In [90]:
fit(m, md, 4, opt, nll_loss_seq)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      2.598523   2.414747  
    1      2.290183   2.198918                              
    2      2.136953   2.08569                               
    3      2.047783   2.012705                              


[array([2.0127])]

In [91]:
 set_lrs(opt, 1e-4)

In [92]:
fit(m, md, 1, opt, nll_loss_seq)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.992547   1.995477  


[array([1.99548])]

#### indentity initiazation

In [62]:
m = CharSeqRnn(vocab_size, n_fac).cuda()
opt = optim.Adam(m.parameters(), 1e-2)

## jeff hinton implementation paper
(A Simple Way to Initialize Recurrent Networks of Rectified Linear Units)
to initialize the weights of the hidden layer with identity matrix 

In [63]:
m.rnn.weight_hh_l0.data.copy_(torch.eye(n_hidden))


    1     0     0  ...      0     0     0
    0     1     0  ...      0     0     0
    0     0     1  ...      0     0     0
       ...          ⋱          ...       
    0     0     0  ...      1     0     0
    0     0     0  ...      0     1     0
    0     0     0  ...      0     0     1
[torch.cuda.FloatTensor of size 256x256 (GPU 0)]

In [96]:
fit(m, md, 4, opt, nll_loss_seq)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      2.327239   2.155149  
    1      2.056674   1.994688                              
    2      1.955719   1.931129                              
    3      1.902586   1.901739                              


[array([1.90174])]

In [97]:
set_lrs(opt, 1e-3)

In [98]:
fit(m, md, 4, opt, nll_loss_seq)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.80646    1.827812  
    1      1.792496   1.821138                              
    2      1.781848   1.816434                              
    3      1.778889   1.812046                              


[array([1.81205])]

In [99]:
set_lrs(opt, 1e-4)

In [100]:
fit(m, md, 4, opt, nll_loss_seq)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.764754   1.805388  
    1      1.764377   1.80452                               
    2      1.762867   1.804002                              
    3      1.760231   1.803494                              


[array([1.80349])]

##### Stateful 

setup

In [64]:
from torchtext import vocab, data

from fastai.nlp import*
from fastai.lm_rnn import *

PATH = ('data/nietzsche/')

TRN_PATH = 'trn/'
VAL_PATH = 'val/'

# TRN_PATH.mkdir(exist_ok=True)
# VAL_PATH.mkdir(exist_ok=True)

TRN = f'{PATH}{TRN_PATH}'
VAL = f'{PATH}{VAL_PATH}'

%ls {PATH}

models/  nietzsche.txt*  trn/  val/


In [65]:
%ls {PATH}trn/

trn.txt*


In [66]:
with open(f'{PATH}nietzsche.txt' , 'r', encoding='utf-8') as f:
    text = f.read()

split = int(len(text) * 0.83529)
trn_txt = text[:split]
val_txt = text[split:]

with open(f'{PATH}trn/trn.txt' , 'w', encoding='utf-8') as f:
    f.write(trn_txt)

with open(f'{PATH}val/val.txt' , 'w', encoding='utf-8') as f:
    f.write(val_txt)

In [67]:
# tokenize based on character, python list gives every character in separeted in a list
list('python')

['p', 'y', 't', 'h', 'o', 'n']

In [69]:
TEXT = data.Field(lower=True, tokenize=list)
bs=64; bptt=8; n_fac=42; n_hidden=256

FILES = dict(train=TRN_PATH, validation=VAL_PATH, test=VAL_PATH)

md = LanguageModelData.from_text_files(PATH, TEXT, **FILES, bs=bs, bptt=bptt, min_freq=2)

len(md.trn_dl), md.nt, len(md.trn_ds), len(md.trn_ds[0].text)
# trn_dl dataloader->how many minibatches ae there
# nt number of unique tokens

(963, 56, 1, 493745)

#### RNN

### Backpropagation through time
After your for loop(e.g this case 8 a.k.a the sequence length) remember the tensor data but through away history of operatioins and start fresh
keep our hidden state but not the hidden state history.

so that during backpropagation it stops there. How many layers to backprop. 
- Best not to BPTT many layers, if you have gradient instability i.e. (explosion or vanishing) the harder to train
- but longer BPTT means able to explicitly capture longer memory or state  
here repackage_var(h) is doing that
###### Choossing BPTT value
- our matrix becomes BPTT by bs (checking this fits ram memory)
- unstable gradient -> adjust bptt cos less layer for the gradients to explode

In [70]:
class CharSeqStatefulRnn(nn.Module):
    def __init__(self, vocab_size, n_fac, bs):
        super().__init__()
        self.vocab_size = vocab_size
        self.e = nn.Embedding(vocab_size, n_fac)
        self.rnn = nn.RNN(n_fac, n_hidden)
        self.l_out = nn.Linear(n_hidden, vocab_size)
        self.init_hidden(bs)

    def forward(self, cs):
        bs = cs[0].size(0)
        #for the last batch(<64),end of epoch or start difference to reinitialize h to match it
        if self.h.size(1) != bs: self.init_hidden(bs) 
        outp, h = self.rnn(self.e(cs), self.h)
        #Backpropagation through time(BPTT)
        self.h = repackage_var(h) # remember the state with out the calculation/operations history, by just taking the tensor data and create new Variable 
        return F.log_softmax(self.l_out(outp), dim=-1).view(-1, self.vocab_size)

    def init_hidden(self, bs): self.h = V(torch.zeros(1, bs, n_hidden))

In [71]:
m = CharSeqStatefulRnn(md.nt, n_fac, 512).cuda()
opt = optim.Adam(m.parameters(), 1e-3)

In [108]:
fit(m, md, 4, opt, F.nll_loss)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                               
    0      1.882646   1.85456   
    1      1.703179   1.699586                               
    2      1.62176    1.629957                               
    3      1.567112   1.58739                                


[array([1.58739])]

In [109]:
set_lrs(opt, 1e-4)

In [110]:
fit(m, md, 4,opt, F.nll_loss)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                               
    0      1.494563   1.544331  
    1      1.483446   1.538085                               
    2      1.491703   1.533786                               
    3      1.482553   1.53072                                


[array([1.53072])]

### RNN Loop

### from pytorch source code
`
def RNNCell(input, hidden, w_ih, w_hh, b_ih, b_hh):  
    return F.tanh(F.linear(input, w_ih, b_ih) + F.linear(hidden, w_hh, b_hh))
    `

In [72]:
class CharSeqStatefulRnn2(nn.Module):
    def __init__(self, vocab_size, n_fac, bs):
        super().__init__()
        self.vocab_size = vocab_size
        self.e = nn.Embedding(vocab_size, n_fac)
        self.rnn = nn.RNNCell(n_fac, n_hidden)
        self.l_out = nn.Linear(n_hidden, vocab_size)
        self.init_hidden(bs)

    def forward(self, cs):
        bs = cs[0].size(0)
        if self.h.size(1) != bs: self.init_hidden(bs) 
        outp=[]
        o= self.h
        for c in cs:
            o = self.rnn(self.e(c), o)
            outp.append(o)
        outp = self.l_out(torch.stack(outp))
        self.h = repackage_var(o) 
        return F.log_softmax(outp, dim=-1).view(-1, self.vocab_size)

    def init_hidden(self, bs): self.h = V(torch.zeros(1, bs, n_hidden))

In [73]:
m = CharSeqStatefulRnn2(md.nt, n_fac, 512).cuda()
opt = optim.Adam(m.parameters(), 1e-3)

In [113]:
fit(m, md, 4, opt, F.nll_loss)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                               
    0      1.882686   1.85789   
    1      1.704045   1.700873                               
    2      1.616389   1.630057                               
    3      1.556519   1.58756                                


[array([1.58756])]

### GRU

In [74]:
class CharSeqStatefulGRU(nn.Module):
    def __init__(self, vocab_size, n_fac, bs):
        super().__init__()
        self.vocab_size = vocab_size
        self.e = nn.Embedding(vocab_size, n_fac)
        self.rnn = nn.GRU(n_fac, n_hidden)
        self.l_out = nn.Linear(n_hidden, vocab_size)
        self.init_hidden(bs)

    def forward(self, cs):
        bs = cs[0].size(0)
        if self.h.size(1) != bs: self.init_hidden(bs)
        outp, h = self.rnn(self.e(cs), self.h)
        self.h = repackage_var(h)
        return F.log_softmax(self.l_out(outp), dim=-1).view(-1, self.vocab_size)

    def init_hidden(self, bs): self.h = V(torch.zeros(1, bs, n_hidden))

In [75]:
#from pytorch source code GRU implementation

def GRUCell(input, hidden, w_ih, w_hh, b_ih, b_hh):
    gi = F.linear(input, w_ih, b_ih)
    gh = F.linear(hidden, w_hh, b_hh)
    i_r, i_i, i_n = gi.chunk(3, 1)
    h_r, h_i, h_n = gh.chunk(3, 1)

    resetgate = F.sigmoid(i_r + h_r)
    inputgate = F.sigmoid(i_i + h_i)
    newgate = F.tanh(i_n + resetgate * h_n)
    return newgate + inputgate * (hidden - newgate)

In [76]:
m = CharSeqStatefulGRU(md.nt, n_fac, 512).cuda()
opt = optim.Adam(m.parameters(), 1e-3)

In [76]:
fit(m, md, 6, opt, F.nll_loss)

Epoch:   0%|          | 0/6 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                               
    0      1.754166   1.724923  
    1      1.570175   1.579178                               
    2      1.486359   1.512956                               
    3      1.436719   1.490934                               
    4      1.401229   1.471287                               
    5      1.359111   1.454863                               


[array([1.45486])]

In [77]:
set_lrs(opt, 1e-4)

In [78]:
fit(m, md, 3, opt, F.nll_loss)

Epoch:   0%|          | 0/3 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                               
    0      1.285352   1.418407  
    1      1.284088   1.414745                               
    2      1.279653   1.412697                               


[array([1.4127])]

#### LSTM

In [86]:
from fastai import sgdr
n_hidden = 512

In [87]:
class CharSeqStatefulLSTM(nn.Module):
    def __init__(self, vocab_size, n_fac, bs, nl):
        super().__init__()
        self.vocab_size, self.nl = vocab_size, nl
        self.e = nn.Embedding(vocab_size, n_fac)
        self.rnn = nn.LSTM(n_fac, n_hidden, nl, dropout=0.5)
        self.l_out = nn.Linear(n_hidden, vocab_size)
        self.init_hidden(bs)

    def forward(self, cs):
        bs = cs[0].size(0)
        if self.h[0].size(1) != bs: self.init_hidden(bs)
        outp, h = self.rnn(self.e(cs), self.h)
        self.h = repackage_var(h)
        return F.log_softmax(self.l_out(outp), dim=-1).view(-1, self.vocab_size)

    def init_hidden(self, bs): 
        self.h = (V(torch.zeros(self.nl, bs, n_hidden)),
                  V(torch.zeros(self.nl, bs, n_hidden)))

In [88]:
m = CharSeqStatefulLSTM(md.nt, n_fac, 512, 2).cuda()
lo = LayerOptimizer(optim.Adam, m, 1e-2, 1e-5)

In [89]:
os.makedirs(f'{PATH}models', exist_ok=True)

In [87]:
fit(m, md, 2, lo.opt, F.nll_loss)

Epoch:   0%|          | 0/2 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.872483   1.780938  
    1      1.725474   1.651992                               


[array([1.65199])]

In [91]:
fit(m, md, 16, lo.opt, F.nll_loss)

Epoch:   0%|          | 0/16 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                               
    0      1.508904   1.480075  
    1      1.505347   1.474703                              
    2      1.507765   1.476236                              
    3      1.509139   1.460136                              
    4      1.513655   1.460003                              
    5      1.497532   1.458086                              
    6      1.500069   1.461942                              
    7      1.489752   1.442052                              
    8      1.501797   1.456596                               
    9      1.497575   1.460863                              
    10     1.486824   1.453607                              
    11     1.495482   1.45353                               
    12     1.487177   1.450065                               
    13     1.499881   1.459459                              
    14     1.490661   1.457816                               
    15     1.485001   1.446623                  

[array([1.44662])]

In [90]:
on_end = lambda sched, cycle: save_model(m, f'{PATH}models/cyc_{cycle}')
cb = [CosAnneal(lo, len(md.trn_dl), cycle_mult=2, on_cycle_end=on_end)]
fit(m, md, 2**4-1, lo.opt, F.nll_loss, callbacks=cb)

Epoch:   0%|          | 0/15 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.546402   1.49858   
    1      1.592816   1.534251                              
    2      1.477493   1.437508                              
    3      1.610668   1.546655                               
    4      1.537422   1.484858                              
    5      1.448801   1.41514                                
    6      1.388635   1.380589                              
    7      1.584235   1.521086                              
    8      1.544689   1.499275                               
    9      1.511404   1.471956                              
    10     1.474738   1.442243                              
    11     1.432706   1.409334                              
    12     1.390985   1.377497                              
    13     1.340526   1.3517                                
    14     1.315724   1.339177                              


[array([1.33918])]

In [81]:
on_end = lambda sched, cycle: save_model(m, f'{PATH}models/cyc_{cycle}')
cb = [CosAnneal(lo, len(md.trn_dl), cycle_mult=2, on_cycle_end=on_end)]
fit(m, md, 2**6-1, lo.opt, F.nll_loss, callbacks=cb)

Epoch:   0%|          | 0/63 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                               
    0      1.612361   1.532047  
    1      1.635286   1.551216                              
    2      1.465337   1.422905                               
    3      1.642531   1.563323                              
    4      1.527082   1.473261                              
    5      1.415425   1.391541                              
    6      1.348842   1.352312                               
    7      1.614355   1.539082                               
    8      1.552345   1.501988                              
    9      1.512401   1.468406                               
    10     1.462924   1.432088                               
    11     1.408506   1.384161                              
    12     1.345226   1.351542                              
    13     1.306932   1.323913                               
    14     1.26303    1.310756                              
    15     1.596495   1.524698               

[array([1.28535])]

#### Test the model

In [92]:
def get_next(inp):
    idxs = TEXT.numericalize(inp)
    p = m(VV(idxs.transpose(0,1)))
    r = torch.multinomial(p[-1].exp(),1)
    return TEXT.vocab.itos[to_np(r)[0]]

In [93]:
get_next('for thos')

'e'

In [94]:
def get_next_n(inp, n):
    res = inp
    for i in range(n):
        c = get_next(inp)
        res += c
        inp = inp[1:]+c
    return res

In [85]:
print(get_next_n('for thos', 400))

for those believex of the wagner--where!--is them likea could duublings:--it stood, conflowingfurnish: perhaps even in the suggestion hazpened utilitrableand in the faculty, auturior, in thegreat explain or in view of threaten powering, and of a life of one'sinterpresy of science, or we should before counter, philosophers havecome to sleepquest, allfoldly, tested ascertainly; what distortivediquent. their


In [95]:
print(get_next_n('for thos', 400))

for those under all mat on the chuglericom,has, preparatively individuity as 'an indeed. "finding according, to divin? and this mereduch proyaged, thinking;but leaveteraris in gent.from things us, the good' beurchrundice, we addition of man were? my rightlessly own "commends, now away, noming learned for the profounding itself thus mashick," as the obsents, (taking,affrance, all disdangest comparison it, 
